In [ ]:
# Problema: Construir una tabla analítica confiable de pedidos y clientes, preservando la relación entre ambas fuentes.

"""Construye la entrega de referencia para PRE_01 desde el archivo plano."""

import csv
import sqlite3
from pathlib import Path


ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
SOURCE = ROOT / "data" / "sales.csv"
OUTPUT = ROOT / "submission" / "sales.db"


def build_database():
    OUTPUT.unlink(missing_ok=True)
    with sqlite3.connect(OUTPUT) as connection, SOURCE.open(newline="", encoding="utf-8") as file:
        connection.execute("PRAGMA foreign_keys = ON")
        connection.executescript(
            """
            CREATE TABLE customers (customer_id TEXT PRIMARY KEY, customer_name TEXT NOT NULL, customer_city TEXT NOT NULL);
            CREATE TABLE orders (order_id TEXT PRIMARY KEY, order_date TEXT NOT NULL, customer_id TEXT NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id));
            CREATE TABLE products (product_id TEXT PRIMARY KEY, product_name TEXT NOT NULL, category TEXT NOT NULL, unit_price REAL NOT NULL, CHECK (unit_price >= 0));
            CREATE TABLE order_items (order_id TEXT NOT NULL, product_id TEXT NOT NULL, quantity INTEGER NOT NULL CHECK (quantity > 0), PRIMARY KEY (order_id, product_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id));
            """
        )
        rows = list(csv.DictReader(file))
        customers = {(r["customer_id"], r["customer_name"], r["customer_city"]) for r in rows}
        orders = {(r["order_id"], r["order_date"], r["customer_id"]) for r in rows}
        products = {(r["product_id"], r["product_name"], r["category"], float(r["unit_price"])) for r in rows}
        items = [(r["order_id"], r["product_id"], int(r["quantity"])) for r in rows]
        connection.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
        connection.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products)
        connection.executemany("INSERT INTO orders VALUES (?, ?, ?)", orders)
        connection.executemany("INSERT INTO order_items VALUES (?, ?, ?)", items)


if __name__ == "__main__":
    build_database()